<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 03 · REAL-TIME ANALYTICS WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Load Data into Apache Doris</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">Match source and delivery requirements to Stream Load, S3 TVF with INSERT INTO SELECT, Routine Load, and Group Commit—then verify quality counts and retry behavior.</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Doris 4.1.3 · Stream Load · Label · S3 TVF · Routine Load · Kafka · Group Commit</span>
</div>

This lab continues with the single-node integrated Doris sandbox created in Lab 1. Its large Parquet source is the same 10,158,080-row ecommerce event dataset used in Labs 1 and 2. The main CSV and nested JSON files contain the same fixed 1,024-record sample taken from that dataset, represented in the format required by each load method.

Stream Load, S3 TVF with `INSERT INTO SELECT`, and Routine Load write to separate target tables. Their row counts and completion evidence therefore remain independent. The Kafka broker is another local container; only the Parquet objects and downloadable files are remote.

Dataset provenance: all clean records come from the public [eCommerce behavior data from multi-category store](https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/data), provided by [REES46](https://rees46.com/en/datasets). A separate ten-row CSV contains one invalid text value in the `revenue` field and produces a rejected transaction.

### Initialize the Lab

Run the next cell before Section 1 and again after every Jupyter kernel restart. It reloads the shared helper, installs the Lab 1/2 output styles, and creates the `lab` object used by later cells. It does not start a container, create a table, download a file, or load data.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT);

## 1. Match workload requirements to a load method

A source type alone is not enough to select a load method. Each workload below exposes six requirements: source, volume, latency, delivery direction, transformation, and retry state.

Drag each available load method onto the matching workload. If dragging is inconvenient, select a method and then click a workload card. Incorrect matches remain unlocked and explain which requirement conflicts with the choice.

In [ ]:
lab.load_method_quiz();

## 2. Isolate the three load methods

This section reconnects to the persisted Doris sandbox and recreates only the three target tables owned by Lab 3. The large Lab 1/2 tables—`events`, `events_v2`, and `events_v3`—are not changed.

All three targets use the same event schema and one Bucket. One Bucket is sufficient for these small local targets; production sizing depends on data volume, BE capacity, and query parallelism. This step creates table structures only; it does not read a source or load any rows. Each load-method section creates its own transaction or job identity when it runs.

In [ ]:
lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")


for table_name in (
    "events_stream", "events_s3", "events_routine"
):
    lab.execute(f"DROP TABLE IF EXISTS {table_name}")

lab.execute("""
CREATE TABLE events_stream (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12, 2) NOT NULL DEFAULT "0.00"
)
DUPLICATE KEY(event_time, event_id, user_id)
DISTRIBUTED BY RANDOM BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")

lab.execute("CREATE TABLE events_s3 LIKE events_stream")
lab.execute("CREATE TABLE events_routine LIKE events_stream")

lab.sql("""
SELECT 'events_stream' AS target_table, COUNT(*) AS row_count FROM events_stream
UNION ALL
SELECT 'events_s3', COUNT(*) FROM events_s3
UNION ALL
SELECT 'events_routine', COUNT(*) FROM events_routine
ORDER BY target_table
""", title="Empty Lab 3 targets");

**Expected result:** the query returns three rows—one for `events_stream`, `events_s3`, and `events_routine`—and every `row_count` is **0**. No CSV, Parquet, or Kafka data has been read or imported yet.

## 3. Load a local CSV with Stream Load

Stream Load pushes bytes from a client-local file. The next cell obtains only the CSV needed for this path and makes both locations visible: the course S3 source and the absolute local path later passed to `curl -T`.

If a valid local copy already exists, it is reused. Otherwise the download is written to a `.part` file, validated, and atomically renamed so that an interrupted transfer cannot replace a valid fixture.

The file contains 1,024 real records sampled from the Lab 1/2 dataset. It uses `|` as its column separator. Doris Stream Load defaults `column_separator` to a tab character, so this request must explicitly set `column_separator:|`; without that header, Doris would parse each line using the wrong field boundary.

The preview shows only the first ten records so that the Notebook remains readable. The complete file is sent by `curl`.

In [ ]:
STREAM_CSV_PATH = "level1/module03-loading-data/downloads/stream_events.csv"

lab.fetch_fixtures([
    (
        "Stream Load CSV",
        "s3://yy-glue-test-us-east-1/doris-course/lab3/v2/stream_events.csv",
        STREAM_CSV_PATH,
        "stream_csv",
    ),
])
lab.preview_fixture(STREAM_CSV_PATH, "stream_csv");

**Expected result:** the source table shows the remote HTTPS URL, the learner's absolute local path, **1,024** rows, and either `downloaded` or `reused`. The preview shows the first ten pipe-delimited records from the same event dataset used in Labs 1 and 2.

The target is truncated before the request, so rerunning this main-load cell performs a real load instead of appending duplicate rows. A fresh label identifies each execution. The request exposes the full Stream Load contract:

- `label` identifies this batch and protects a retry of the same batch.
- `column_separator:|` is required because this CSV uses pipes rather than commas between fields.
- `columns` first names the text values `raw_event_time` and `raw_revenue`, then maps them to the target columns.
- `STR_TO_DATE` converts the CSV time text to `DATETIME`, and `CAST` converts revenue text to `DECIMAL(12,2)`.
- `strict_mode:true` prevents an invalid conversion from being silently accepted.

In [ ]:
lab.execute("TRUNCATE TABLE events_stream")
STREAM_LABEL = f"module3_csv_{lab.new_run_suffix()}"

STREAM_REQUEST = f"""
curl --silent --show-error --location-trusted -u root: \
  -H "Expect:100-continue" \
  -H "label:{STREAM_LABEL}" \
  -H "format:csv" \
  -H "column_separator:|" \
  -H "strict_mode:true" \
  -H "max_filter_ratio:0" \
  -H "columns:raw_event_time,event_id,user_id,event_type,region,product_id,raw_revenue,event_time=str_to_date(raw_event_time,'%Y-%m-%d %H:%i:%s'),revenue=cast(raw_revenue as decimal(12,2))" \
  -T "{STREAM_CSV_PATH}" \
  -X PUT http://127.0.0.1:8030/api/doris_course/events_stream/_stream_load
"""

stream_response = lab.stream_load(
    STREAM_REQUEST,
    title="Stream Load result",
    expected_status="Success",
    expected_counts={
        "NumberTotalRows": 1024,
        "NumberLoadedRows": 1024,
        "NumberFilteredRows": 0,
        "NumberUnselectedRows": 0,
    },
)

lab.sql("""
SELECT
    COUNT(*) AS loaded_rows,
    COUNT(DISTINCT event_id) AS distinct_events,
    MIN(event_time) AS min_event_time,
    MAX(event_time) AS max_event_time,
    SUM(revenue) AS total_revenue
FROM events_stream
""", title="Committed Stream Load sample");

**Expected result**

The response reports `1,024 total = 1,024 loaded + 0 filtered + 0 unselected`. The validation query returns:

| loaded_rows | distinct_events | min_event_time | max_event_time | total_revenue |
|---:|---:|---|---|---:|
| 1024 | 1024 | 2019-12-01 03:08:44 | 2020-03-03 12:12:47 | 23299.84 |

`Status=Success` means the batch transaction committed. The result also shows the request's elapsed time; `TxnId` and timings vary. The complete response remains available in the collapsed JSON output, including `ErrorURL` when Doris provides one.

### Retry the uncertain request with the same label

A client can lose the response after Doris commits. Reissuing the same bytes with the same label asks about the same batch; it must not create another transaction or append another 1,024 rows to this Duplicate Key model table.

For this retry, the meaningful response fields are `status` and `label`. Doris rejects the duplicate label before starting another load, so no new transaction or row-quality result exists. The table count below supplies the decisive evidence that no rows were appended.

In [ ]:
retry_response = lab.stream_load(
    STREAM_REQUEST,
    title="Same-label retry",
    expected_status="Label Already Exists",
    display_columns=["status", "label"],
)

lab.sql(
    "SELECT COUNT(*) AS rows_after_retry FROM events_stream",
    title="Rows after same-label retry",
);

**Expected result**

`Same-label retry` contains only the evidence relevant to retry protection:

| status | label |
|---|---|
| Label Already Exists | The same `module3_csv_...` label used by the successful request |

`Rows after same-label retry` returns:

| rows_after_retry |
|---:|
| 1,024 |

The unchanged table count proves that Doris did not submit a second transaction. A different label would identify a new transaction and would append rows to this Duplicate Key model table.

### Reject a batch that exceeds its quality threshold

The isolated ten-row CSV is copied from the Lab 1/2 dataset, with only the final `revenue` value replaced by `not-a-number`. Stream Load reports the row-level parsing failure and does not commit a partially accepted batch.

The request uses a new label and `max_filter_ratio:0`. One conversion failure therefore exceeds the allowed error ratio and rejects the entire transaction. Doris returns an `ErrorURL` in the Stream Load JSON response. The code extracts that field, immediately sends an HTTP GET request to it, and displays the response without converting it into a custom table. Error logs are temporary diagnostic resources, so inspect them immediately after the failed request.

In [ ]:
MALFORMED_CSV_PATH = "level1/module03-loading-data/downloads/stream_events_malformed.csv"

lab.fetch_fixtures([
    (
        "Malformed Stream Load CSV",
        "s3://yy-glue-test-us-east-1/doris-course/lab3/v2/stream_events_malformed.csv",
        MALFORMED_CSV_PATH,
        "stream_malformed_csv",
    ),
])

REJECT_LABEL = f"module3_csv_reject_{lab.new_run_suffix()}"

REJECT_REQUEST = f"""
curl --silent --show-error --location-trusted -u root: \
  -H "Expect:100-continue" \
  -H "label:{REJECT_LABEL}" \
  -H "format:csv" \
  -H "column_separator:|" \
  -H "strict_mode:true" \
  -H "max_filter_ratio:0" \
  -H "columns:raw_event_time,event_id,user_id,event_type,region,product_id,raw_revenue,event_time=str_to_date(raw_event_time,'%Y-%m-%d %H:%i:%s'),revenue=cast(raw_revenue as decimal(12,2))" \
  -T "{MALFORMED_CSV_PATH}" \
  -X PUT http://127.0.0.1:8030/api/doris_course/events_stream/_stream_load
"""

reject_response = lab.stream_load(
    REJECT_REQUEST,
    title="Rejected Stream Load batch",
    expected_status="Fail",
    expected_counts={"NumberTotalRows": 10, "NumberFilteredRows": 1},
    show_response=False,
    show_json=False,
)
error_url = reject_response["ErrorURL"]
lab.show_stream_load_error_log(error_url);

**Expected result**

- **Obtain the URL:** `reject_response` is the JSON response returned by Stream Load. `reject_response["ErrorURL"]` reads its `ErrorURL` field and displays the original URL generated by Doris.
- **Retrieve the diagnostic:** the notebook immediately performs an HTTP GET on that URL and displays the original response under `Content returned by ErrorURL`. The equivalent manual operation is `curl "<ErrorURL>"` while the temporary log is still available.
- **Locate the rejected record:** find the source record containing `not-a-number`. Its pipe-delimited fields follow `event_time | event_id | user_id | event_type | region | product_id | revenue`, so the invalid value belongs to `revenue`.
- **Read the reason:** the diagnostic explains that the source value cannot be converted to the target `DECIMAL(12,2)` under strict mode. Doris counts this as one filtered row.
- **Understand the transaction result:** the response is checked as `Status=Fail`, `NumberTotalRows=10`, `NumberLoadedRows=0`, and `NumberFilteredRows=1`. Because `max_filter_ratio` is `0`, the complete ten-row transaction is rejected and no row from this request is committed.

## 4. Load Parquet with S3 TVF and INSERT INTO SELECT

The S3 TVF reads the same 40 remote Parquet objects used in Lab 1 and presents them as a temporary relation. A TVF query does not persist rows. Immediately before loading, `TRUNCATE TABLE` removes earlier rows. `INSERT INTO SELECT` then performs the durable, synchronous copy into `events_s3` and reports the actual insertion time. The helper applies temporary low-memory S3 concurrency settings and restores them after the statement finishes.

Remote object pattern: `s3://yy-glue-test-us-east-1/doris-course/events/v1/events-*.parquet`

The SQL writes the complete S3 TVF directly, including the object URI, endpoint, region, and Parquet format. Only the two credential values remain as named placeholders. The notebook reads them from `course_secrets.env` and substitutes them in memory at execution time; secret values are never written into the notebook.

In [ ]:
if not lab.load_s3_credentials():
    raise RuntimeError("Complete course_secrets.env before continuing.")

lab.sql("""
SELECT event_time, event_id, user_id, event_type, region, product_id, revenue
FROM S3(
    "uri" = "s3://yy-glue-test-us-east-1/doris-course/events/v1/events-*.parquet",
    "s3.endpoint" = "https://s3.us-east-1.amazonaws.com",
    "s3.region" = "us-east-1",
    "s3.access_key" = "{{S3_READ_ONLY_ACCESS_KEY}}",
    "s3.secret_key" = "{{S3_READ_ONLY_SECRET_KEY}}",
    "format" = "parquet"
)
LIMIT 5
""", title="Remote Parquet preview")

lab.sql(
    "SELECT COUNT(*) AS rows_before_insert FROM events_s3",
    title="Internal target before INSERT",
);

**Expected result:** five remote event rows are visible while `rows_before_insert` remains **0**. This proves that querying the TVF and storing data in an internal table are separate operations.

In [ ]:
lab.execute("TRUNCATE TABLE events_s3")

lab.insert("""
INSERT INTO events_s3 (
    event_time, event_id, user_id, event_type, region, product_id, revenue
)
SELECT
    event_time, event_id, user_id, event_type, region, product_id, revenue
FROM S3(
    "uri" = "s3://yy-glue-test-us-east-1/doris-course/events/v1/events-*.parquet",
    "s3.endpoint" = "https://s3.us-east-1.amazonaws.com",
    "s3.region" = "us-east-1",
    "s3.access_key" = "{{S3_READ_ONLY_ACCESS_KEY}}",
    "s3.secret_key" = "{{S3_READ_ONLY_SECRET_KEY}}",
    "format" = "parquet"
)
""", title="S3 Parquet insert", low_memory_s3=True)

lab.sql("""
SELECT
    COUNT(*) AS target_rows,
    MIN(event_time) AS min_event_time,
    MAX(event_time) AS max_event_time,
    SUM(revenue) AS total_revenue
FROM events_s3
""", title="S3 batch validation");

**Expected result:** the insert reports **10,158,080 affected rows** and its elapsed time. `events_s3` then contains **10,158,080** rows from `2019-12-01 00:00:12` through `2020-03-11 04:53:08`, with total revenue **39,984,455.64**.

Unlike Stream Load, this statement has no client-supplied label. The target is explicitly truncated before each execution, so the notebook performs a complete replacement and exposes the actual insertion time. A production workflow should instead use an orchestrator and an idempotent design appropriate to its availability requirements.

## 5. Continuously load Kafka JSON with Routine Load

Routine Load is appropriate when the source is not a bounded file but a Kafka topic that continues to receive messages. A Routine Load job is a Doris-managed, long-running Kafka consumer job. Doris schedules its micro-batch tasks, commits their data and Kafka offsets together, and maintains the job state.

The included Compose file starts one Apache Kafka 3.9.1 broker in KRaft mode on the existing `doris-course` network. Docker selects the native `linux/arm64` or `linux/amd64` image automatically. The addresses serve different callers:

| Caller | Broker address |
|---|---|
| Doris BE inside the Docker network | `kafka:29092` |
| A client running on the host | `localhost:9092` |

The JSON Lines fixture is obtained in this section because the Kafka console producer is its first consumer. It contains the same 1,024 logical records used by Stream Load, represented as nested JSON. The output shows the course S3 source and the local path later supplied through shell input redirection.

This subsection is independently repeatable after `events_routine` has been created once in Section 2. Run its three code cells in order:

```text
1. Prepare the Kafka source
   ├── create a new Kafka topic
   ├── choose the Routine Load job name
   └── do not create the Doris job yet

2. Create the Routine Load job and mapping
   ├── define how nested JSON fields map to table columns
   ├── execute CREATE ROUTINE LOAD for the chosen topic
   └── wait until the Doris job is RUNNING

3. Publish and observe
   ├── publish JSON messages to the Kafka topic
   └── observe Doris consume and commit each batch
```

The second cell is required before publication: the first cell creates the Kafka topic and reserves a job name, but only `CREATE ROUTINE LOAD` creates the Routine Load job. Rerunning the Kafka preparation cell stops its previous job if necessary and selects a new topic, consumer group, and job name. Candidate names become active notebook variables only after Kafka preparation succeeds, so an interrupted preparation cannot leave later cells pointing at an uncreated topic. The Routine Load job cell truncates `events_routine` before creating the new job. Kafka messages and committed offsets therefore cannot leak from the previous execution into the new one.

In [ ]:
lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")

previous_routine_job = globals().get("ROUTINE_JOB")

next_routine_run_suffix = lab.new_run_suffix()
next_kafka_topic = f"doris-events-json-{next_routine_run_suffix.replace('_', '-')}"
next_routine_job = f"module3_events_json_{next_routine_run_suffix}"

lab.stop_routine_if_exists(previous_routine_job)

ROUTINE_JSON_PATH = "level1/module03-loading-data/downloads/routine_events.jsonl"

lab.fetch_fixtures([
    (
        "Routine Load JSON Lines",
        "s3://yy-glue-test-us-east-1/doris-course/lab3/v2/routine_events.jsonl",
        ROUTINE_JSON_PATH,
        "routine_json",
    ),
])

lab.shell(fr"""
set -euo pipefail

echo "[1/3] Start the pinned single-broker Kafka sandbox"
docker compose -f level1/module03-loading-data/compose.kafka.yml up -d

echo "[2/3] Wait for the Kafka health check"
for attempt in $(seq 1 60); do
  STATUS=$(docker inspect --format '{{{{if .State.Health}}}}{{{{.State.Health.Status}}}}{{{{else}}}}none{{{{end}}}}' kafka)
  [ "$STATUS" = "healthy" ] && break
  [ "$attempt" -eq 60 ] && {{ docker logs --tail 100 kafka; exit 1; }}
  sleep 2
done

echo "[3/3] Create an isolated topic for this Lab run"
docker exec kafka /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server kafka:29092 \
  --create --if-not-exists \
  --topic "{next_kafka_topic}" \
  --partitions 1 \
  --replication-factor 1

docker compose -f level1/module03-loading-data/compose.kafka.yml ps
""", title="Prepare the Kafka source")

lab.preview_fixture(ROUTINE_JSON_PATH, "routine_json")

ROUTINE_RUN_SUFFIX = next_routine_run_suffix
KAFKA_TOPIC = next_kafka_topic
ROUTINE_JOB = next_routine_job;

**Expected result:** Kafka becomes `healthy`, the unique topic is created with one partition, and the source table reports **1,024** JSON messages. The preview shows the first ten nested representations of the same clean records used by Stream Load. At this point the Kafka source exists, but the Doris Routine Load job does not; run the next cell before publishing messages.

The Kafka volume is persistent, but the unique topic prevents messages from an older Lab run from entering this run. The producer cell below also refuses an ambiguous partially populated topic.

### Create the Routine Load job and define its JSON mapping

This cell creates the Routine Load job required by the following publish cell. `CREATE ROUTINE LOAD` connects the chosen job name to the Kafka topic, while `jsonpaths` extracts seven values from each nested message in a fixed order. `COLUMNS` maps those values to target columns or temporary names: the ISO time string is converted to `DATETIME`, and the uppercase JSON event type is converted back to the lowercase value used in the Lab 1/2 dataset. The numeric JSON revenue maps to the target `DECIMAL(12,2)`.

All 1,024 messages are valid, so any difference between published messages and committed target rows reflects consumption progress rather than row-quality filtering. The target is truncated immediately before the job is created, ensuring that all rows observed afterward belong to this execution.

In [ ]:
lab.execute("TRUNCATE TABLE events_routine")

lab.execute(fr"""
CREATE ROUTINE LOAD doris_course.{ROUTINE_JOB} ON events_routine
COLUMNS(
    raw_event_time,
    event_id,
    user_id,
    raw_event_type,
    raw_region,
    product_id,
    revenue,
    event_time = STR_TO_DATE(raw_event_time, '%Y-%m-%dT%H:%i:%s'),
    event_type = LOWER(raw_event_type),
    region = raw_region
)
PROPERTIES (
    "format" = "json",
    "jsonpaths" = "[\"$.meta.occurred_at\",\"$.meta.event_id\",\"$.actor.user_id\",\"$.event.type\",\"$.geo.region\",\"$.product_id\",\"$.event.revenue\"]",
    "desired_concurrent_number" = "1",
    "max_batch_interval" = "5",
    "strict_mode" = "true",
    "max_filter_ratio" = "0"
)
FROM KAFKA (
    "kafka_broker_list" = "kafka:29092",
    "kafka_topic" = "{KAFKA_TOPIC}",
    "property.kafka_default_offsets" = "OFFSET_BEGINNING",
    "property.group.id" = "{ROUTINE_JOB}"
)
""")

lab.wait_for_routine_state(ROUTINE_JOB, "RUNNING")
lab.routine_load_status(ROUTINE_JOB);

**Expected result:** the helper waits through the temporary `NEED_SCHEDULE` state and displays the job only after it reaches `RUNNING`. This confirms that the Routine Load job now exists and is ready for the following publish cell. Before messages are published, the total, loaded, and error counters are zero.

### Publish messages and observe committed progress

The producer sends each JSON line as one Kafka message. It publishes the 1,024 messages in four batches of 256. After each batch, the result records the Kafka and Doris counters immediately, then refreshes them every second until Routine Load commits that checkpoint. The producer waits two seconds before publishing the next batch.

The live pipeline is updated in place as `Notebook producer → Kafka topic → Routine Load job → events_routine`. It makes the temporary queue backlog explicit: `waiting for Routine Load = Kafka published − Routine Load committed`. Kafka retains consumed messages according to its retention policy, so this value means messages not yet committed by the Routine Load job; it does not mean consumed messages were physically deleted from Kafka.

The history below the pipeline names the actor in every action. `Notebook producer · published 256` means the producer has added the first batch to Kafka. `Routine Load job · waiting for 256` is a one-second poll while those messages remain uncommitted. `Routine Load job · committed 256` means the job has committed that checkpoint and the rows are queryable in `events_routine`.

In [ ]:
lab.publish_kafka_batches(
    topic=KAFKA_TOPIC,
    fixture_path=ROUTINE_JSON_PATH,
    job_name=ROUTINE_JOB,
    target_table="events_routine",
    batch_size=256,
    poll_interval_seconds=1,
    batch_pause_seconds=2,
)

lab.sql("""
SELECT
    COUNT(*) AS loaded_rows,
    COUNT(DISTINCT event_id) AS distinct_events,
    MIN(event_time) AS min_event_time,
    MAX(event_time) AS max_event_time,
    SUM(revenue) AS total_revenue
FROM events_routine
""", title="Committed Routine Load sample");

**Expected result**

The live pipeline begins with zero published messages, zero waiting messages, and zero Doris rows. For each batch, `Kafka published` first increases by 256 while `Routine Load committed` and `events_routine rows` may still show the previous checkpoint. During this interval, `waiting for Routine Load` is 256. Each batch then contributes a `Routine Load job · committed` row at 256, 512, 768, and 1,024 rows. At the final checkpoint:

- `Kafka published`, `Routine Load committed`, and `events_routine rows` are all 1,024.
- `waiting for Routine Load` is 0, `errors` remains 0, and `Routine Load state` remains `RUNNING`.
- The validation query reports 1,024 distinct event IDs, a time range from `2019-12-01 03:08:44` through `2020-03-06 03:10:18`, and total revenue **23,299.84**.

The exact number of `Routine Load job · waiting` snapshots depends on the local containers. A fast Routine Load job may commit before the first one-second poll, but the separate producer and Routine Load job rows still identify who performed each action. The two-second gap between batches is instructional pacing; the counters and commits are real, not a simulated progress animation.

### Pause and resume the Routine Load job

Pause stops new Routine Load tasks without deleting the job, committed offsets, or target rows. Resume returns the job through `NEED_SCHEDULE` to `RUNNING`; already committed Kafka messages are not consumed again.

In [ ]:
lab.execute(f"PAUSE ROUTINE LOAD FOR {ROUTINE_JOB}")
lab.wait_for_routine_state(ROUTINE_JOB, "PAUSED")
lab.routine_load_status(ROUTINE_JOB);

**Expected result:** the job state is `PAUSED`; all 1,024 committed rows remain queryable.

In [ ]:
lab.execute(f"RESUME ROUTINE LOAD FOR {ROUTINE_JOB}")
lab.wait_for_routine_state(ROUTINE_JOB, "RUNNING")
lab.routine_load_status(ROUTINE_JOB);

**Expected result:** the job returns to `RUNNING` with the same 1,024-message progress and 1,024 target rows. Publishing the same JSON again would create 1,024 new Kafka messages; that is a new write, not a Routine Load retry.

### Stop the Routine Load job permanently

Run this cleanup cell after completing the pause/resume observation. `STOP` is terminal: the job cannot be resumed, but all rows already committed to `events_routine` remain in Doris.

In [ ]:
lab.execute(f"STOP ROUTINE LOAD FOR {ROUTINE_JOB}")
lab.wait_for_routine_state(ROUTINE_JOB, "STOPPED")
lab.routine_load_status(ROUTINE_JOB);

**Expected result:** the job state is `STOPPED`; the 1,024 target rows remain queryable.

### Stop the Kafka sandbox

Run this optional cell when you want to release the Kafka process and host port 9092. The `kafka-data` named volume and all Doris tables remain available.

In [ ]:
lab.shell(r"""
set -euo pipefail

docker compose -f level1/module03-loading-data/compose.kafka.yml stop kafka
docker inspect --format 'container={{.State.Status}}' kafka
""", title="Stop the Kafka sandbox");

**Expected result:** Docker reports `container=exited` for Kafka.

### Restart the Kafka sandbox

Run this cell when you want to reuse the existing broker and topics. It starts or reuses the existing Kafka container and waits for its health check. A Routine Load job that was permanently stopped remains stopped.

This restart cell is idempotent: it can be run when Docker Desktop and the container are stopped, starting, or already running. On macOS it opens Docker Desktop when necessary; on Linux, start Docker Engine before running the cell.


In [ ]:
lab.start_container("kafka")

**Expected result:** Docker reports `container=running health=healthy`.

## Lab complete

You selected load methods from workload requirements, loaded a real event sample through Stream Load, proved same-label retry safety and transaction-level rejection with an isolated malformed file, separated S3 TVF inspection from an atomic `INSERT INTO SELECT`, and loaded the same logical sample from nested Kafka JSON through a Routine Load job.

The main distinctions are:

- Stream Load is a bounded client push with a synchronous JSON response and label-based duplicate protection.
- An S3 TVF exposes remote files as a temporary relation; `INSERT INTO SELECT` persists an atomic batch.
- A Routine Load job is a Doris-managed, long-running Kafka consumer job whose micro-batch transactions commit data and offsets together.
- Group Commit optimizes frequent small Stream Load or INSERT writes; it does not replace an object-storage or Kafka connector.
- A filtered row and an entirely rejected transaction are different outcomes: the row fails conversion first, then the quality threshold determines whether the batch commits.

Official references: [Load overview](https://doris.apache.org/docs/4.x/data-operate/import/load-manual/) · [Stream Load](https://doris.apache.org/docs/4.x/data-operate/import/import-way/stream-load-manual/) · [INSERT INTO SELECT](https://doris.apache.org/docs/4.x/data-operate/import/import-way/insert-into-manual/) · [Routine Load](https://doris.apache.org/docs/4.x/data-operate/import/import-way/routine-load-manual/) · [Group Commit](https://doris.apache.org/docs/4.x/data-operate/import/load-best-practices/group-commit-manual/) · [Apache Kafka Docker image](https://kafka.apache.org/39/getting-started/docker/)